# 03-topic-divergence

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/news/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('news'))


In [ ]:
%pprint

# Regular Libraris
import pickle
import os
import re     
import time
import datetime
import threading    
import urllib.request 

import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as img

from tqdm import tqdm   
from scipy import stats, sparse
from scipy.special import rel_entr
from datetime import datetime
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor

# ntlk
import nltk
from nltk.corpus import stopwords as stpw     
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer 

# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import GridSearchCV

# gensim
import gensim
import gensim.corpora as corpora
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel, LdaModel

# LDA Visualiztion
import pyLDAvis
import pyLDAvis.lda_model

# Locate folder
os.chdir(workspace('news'))
path = workspace('news')      # Change path here

save_data = True
save_plot = True

In [ ]:
# Additional stopwords
stopwords = ['county','pennsylvania','virginia','additional','ohio','bringing',
             'week','year','fall','countywide','area','state','report','statewide',
            'amid', 'pittsburgh','covid','district','announces','west','local','center',
            'day','phase','virus','plan','beaver','pandemic','north','washington','link',
            'july','april','announced','percent','reporting','probable','month','thing','lot',
            'hour','hill','penn','city','bull','wear','wearing','staff','including','told',
            'august','decision','group','bar','high','stand','increase','june','needed','start',
            'today','feel','pennsylvanian','team','entire']


# Read data
cbs       = pd.read_csv('./Data/CBS_KDKA/cbs_clean.csv', encoding = "utf_8_sig")
cbs_list1 = pd.read_csv('./Data/CBS_KDKA/cbs_title.csv', encoding = "utf_8_sig")
cbs_list2 = pd.read_csv('./Data/CBS_KDKA/cbs_keywords.csv', encoding = "utf_8_sig")
cbs_list3 = pd.read_csv('./Data/CBS_KDKA/cbs_content.csv', encoding = "utf_8_sig")
cbs_list4 = pd.read_csv('./Data/CBS_KDKA/cbs_selectedwords.csv', encoding = "utf_8_sig")

cbs_list1.set_index('Date', inplace = True) # Title
cbs_list2.set_index('Date', inplace = True) # Keywords
cbs_list3.set_index('Date', inplace = True) # Content
cbs_list4.set_index('Date', inplace = True) # Selectedwords

days = list(cbs_list1.index)

# Get (daily) document set
documents1 = list(cbs_list1.iloc[:,0])
documents2 = list(cbs_list2.iloc[:,0])
documents3 = list(cbs_list3.iloc[:,0])
documents4 = list(cbs_list4.iloc[:,0])
documents = [documents1, documents2, documents3, documents4]


# Load the saved model
LDAModels = []
tfs = []
tf_vectorizers = []
keyterms = []
n_maxs = [0.8,0.8,0.8,0.8] #[219, 64, 1731, 649]

for i in range(4) :
    filename = './Data/CBS_KDKA/LDA/adjusted_LDA%d.pkl'%(i+1)
    with open(filename, 'rb') as file:
        LDAModels.append(pickle.load(file))  

    # Prepare vectorized document
    n_keyterms = 50     # Number of key terms selected
    n_min = 10          # Ignore terms that have a document frequency strictly lower than the given threshold
    n_max = n_maxs[i]   # Ignore terms that have a document frequency strictly higher than the given threshold

    vectorizer = CountVectorizer(max_features = n_keyterms , 
                                    min_df = n_min , max_df = n_max , stop_words=stopwords  )
    
    tf_vectorizers.append(vectorizer)
    tf = vectorizer.fit_transform( documents[i] )
    tfs.append(tf)
    keyterms.append(vectorizer.get_feature_names_out()) # Get keyterms

In [ ]:
# Read Daily Topic Percent data
cbs_dtp_list1 = pd.read_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent1.csv', encoding = "utf_8_sig")
cbs_dtp_list2 = pd.read_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent2.csv', encoding = "utf_8_sig")
cbs_dtp_list3 = pd.read_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent3.csv', encoding = "utf_8_sig")
cbs_dtp_list4 = pd.read_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent4.csv', encoding = "utf_8_sig")

cbs_dtp_list1.set_index('Date',inplace = True)
cbs_dtp_list2.set_index('Date',inplace = True)
cbs_dtp_list3.set_index('Date',inplace = True)
cbs_dtp_list4.set_index('Date',inplace = True)

cbs_dtp1 = cbs_dtp_list1.iloc[:,1:]
cbs_dtp2 = cbs_dtp_list2.iloc[:,1:] 
cbs_dtp3 = cbs_dtp_list3.iloc[:,1:]
cbs_dtp4 = cbs_dtp_list4.iloc[:,1:]
dtps = [cbs_dtp1,cbs_dtp2,cbs_dtp3,cbs_dtp4]
topicNums = [len(dtp.columns) for dtp in dtps]

# Static LDA K-L Divergence

## Functions 

In [ ]:
# KL-Divergence: Smooth
##########################################################################
def forward_smooth_tf( tf , smooth_period ):
    
    '''
        smooth_period :  n>=0    distributions of [-n,0] 
        smooth_period :    -1    distributions of [..,0]   '''    
    
    M = pd.DataFrame(np.array(tf.todense()))
    m, n = M.shape[0], M.shape[1]
    N = M.copy()
    N.iloc[1:,:] = 0

    if smooth_period==-1:
        N = M.cumsum()
    elif smooth_period>0:
        N.iloc[:smooth_period,:] = M.iloc[:smooth_period,:].cumsum()
        for row in range( smooth_period , m ):
            N.iloc[row,:] = M.iloc[row-smooth_period:row+1,:].sum()
    elif smooth_period==0:
        N = M
    else:
        raise ValueError('smooth_period should be a nonegative int or -1')
        return
    
    N = sparse.csr_matrix(N.values)     
    return N


##########################################################################
def ForwardSmoothDist(  tf, model, topicNum, days, smooth_period ):
    
    topics = ['Topic %d'%(x+1) for x in range(topicNum)]   
    
    tf = forward_smooth_tf( tf , smooth_period )
    dist = model.transform(tf)
    df = pd.DataFrame(dist,index=days,columns=topics)
    
    for topic in topics:
        df[topic]=pd.to_numeric(df[topic], errors='coerce')
            
    return df

In [ ]:
# KL-Divergence: K-L Divergence Time Series
def KLDseries( dtp, cal_period = 7 ):
    
    '''
    dtp daily topic percentage is a data frame in the form of:
    
    index        Topic 1    Topic 2      Topic 3     Topic 4
    2020/03/02	0.242719	0.242719	0.242719	0.271843
    2020/03/03	0.223216	0.223216	0.250000	0.303568
    2020/03/05	0.250000	0.201615	0.225808	0.322576
    2020/03/06	0.310809	0.168923	0.229731	0.290538
    2020/03/07	0.331322	0.150606	0.204821	0.313251
       ...         ...        ...          ...        ...
    2020/08/27	0.291834	0.267066	0.193263	0.247837
    2020/08/28	0.290375	0.266723	0.195609	0.247293
    2020/08/29	0.288932	0.266702	0.198244	0.246122
    2020/08/30	0.288280	0.265840	0.200214	0.245666
    2020/08/31	0.287777	0.266031	0.200278	0.245914
    
    '''
    
    period = cal_period
    dates = list(dtp.index)
    days  = list(dtp.index[1:])
    
    KLD = []
    
    if period == -1: 
        for day in days:
            dayindex = dates.index(day)
            temp = [sum( rel_entr( list(dtp.iloc[dayindex,:]), 
                                       list(dtp.iloc[i,:]) ) ) 
                        for i in range(dayindex)]
            KLD.append(sum(temp)/len(temp))  
    
    elif period > 0: 
        
        for day in days:
            dayindex = dates.index(day)
            
            if dayindex < period:
                temp = [sum( rel_entr( list(dtp.iloc[dayindex,:]), 
                                           list(dtp.iloc[i,:]) ) ) 
                            for i in range(dayindex)]
                KLD.append(sum(temp)/len(temp))
            
            else:
                temp = [sum( rel_entr( dtp.iloc[dayindex,:], 
                                   dtp.iloc[i,:]) ) 
                    for i in range(dayindex-period,dayindex)]
                KLD.append(sum(temp)/len(temp)) 
    
    else:
        raise ValueError('cal_period should be a positive int or -1')
        return
    
    
    KLDdf = pd.DataFrame(KLD,index=days,columns=['Daily KLD'])
    return KLDdf

In [ ]:
# Visualization

######################################### Heat map ###########################################################
def topic_Change_heatmap( dtp, paraSets, delta = 0.1 , displaybar = True): 
    
    # delta is the row label coordinate
    # paraSet = [doctypeindex / DocNum, smooth_period, cal_period ]
    
    # doctypeindex = [1,2,3,4]
    doctype = ['Title','Keywords','Content','Title and Keywords']
    
    dtp_t = dtp.T
    n = len(dtp.index)
    topicNum = len(dtp.columns)
    
    maxVal = dtp_t.max().max()
    minVal = dtp_t.min().min()   
    
    # fig = plt.figure(figsize = (20,5))
    h = sns.heatmap(dtp_t, center = (maxVal+minVal)/2, xticklabels = 4, cmap = "rocket_r", cbar=displaybar)
    
    # Title
    cpy = paraSets.copy()
    cpy[1]  = '%d'%paraSets[1]
    if cpy[1]=='-1':
        cpy[1]='formerAll'
         
    titlestr = (doctype[cpy[0]-1] + ' : smooth_period = ' + cpy[1])
    plt.title(titlestr, fontproperties = 'Times New Roman', fontsize = 19, fontweight="bold")
    
    # Ticks and Labels
    plt.yticks(np.arange(topicNum) + delta, dtp_t.index,
               fontproperties = 'Times New Roman', size = 15, fontweight="bold");
    plt.xticks(fontproperties = 'Times New Roman', size = 15, fontweight="bold");
    plt.xlabel('',fontproperties = 'Times New Roman',fontsize = 25,fontweight="bold")
    
    return h




######################################### KLDtrend ###########################################################
def plot_KLDtrend( KLDSer, paraSets , step = 4 ):
    # step = xtickstep
    df = KLDSer
    n = len(df.index)
    daily_index = np.arange(n)
    values = df.iloc[:,0]
    # fig = plt.figure( figsize = (16,10) ) 

    plt.grid(linestyle = '-', linewidth = 1.5)
    h = plt.plot ( daily_index ,  values , 
              linewidth = 2 , linestyle = '-' , color = (0, 95/255, 115/255)) 

    # Title
    cpy = paraSets.copy()
    doctype = ['Title','Keywords','Content','Title and Keywords']
    cpy[1] , cpy[2]  = '%d'%paraSets[1] , '%d'%paraSets[2]
    if cpy[1]=='-1':
        cpy[1]='formerAll'
    if cpy[2]=='-1':
        cpy[2]='formerAll'       
    titlestr = ('Local News Daily K-L Divergence Trending\n' + doctype[cpy[0]-1] + ' : smooth_period = ' + cpy[1] 
                +'  &  cal_period = '+ cpy[2])
    
    plt.title(titlestr, fontproperties = 'Times New Roman', fontsize = 19, fontweight="bold")
    
    plt.xticks(daily_index[0:n:step] , df.index[0:n:step] , 
               fontproperties = 'Times New Roman' , fontsize = 15, 
               rotation = 90, fontweight="bold");
    plt.yticks(fontproperties = 'Times New Roman', size = 15, fontweight="bold");
    
    plt.xlim([-0.5,179.5])
    
    return h

######################################### displayKLD ###########################################################
def displayKLD( DocNum, smooth_period, cal_period , para = [13 , 15, 0.1, 3] , figsize = (10,5) ):
    
    # para = [ysize, tsize, delta, step]
    # ysize = ylabel size
    # tsize = title size
    # delta = heatmap ylabel coordinate
    # step = common xtick step
    
    ysize,tsize,delta,step = para[0],para[1],para[2],para[3]

    
    global dtps, tfs, LDAModels, topicNums, days
    paraSets = [DocNum,smooth_period,cal_period]
    
    # Title
    cpy = paraSets.copy()
    doctype = ['Title','Keywords','Content','Title and Keywords']
    cpy[1] , cpy[2]  = '%d'%paraSets[1] , '%d'%paraSets[2]
    if cpy[1]=='-1':
        cpy[1]='formerAll'
    if cpy[2]=='-1':
        cpy[2]='formerAll'       
    titlestr = ('Local News Daily K-L Divergence Trending\n' + doctype[cpy[0]-1] + ' : smooth_period = ' + cpy[1] 
                +'  &  cal_period = '+ cpy[2])
    
    smt = ForwardSmoothDist( tfs[DocNum-1], LDAModels[DocNum-1], topicNums[DocNum-1], days, smooth_period)
    KLDSer = KLDseries(smt, cal_period)

    fig = plt.figure( figsize = figsize ) 
    grid = plt.GridSpec(36, 10, wspace=0.5, hspace=0.5);
    
    ############### heat1 ############
    plt.subplot(grid[0:12,:])
    h1 = topic_Change_heatmap(dtps[DocNum-1].iloc[1:,:], [DocNum,0], delta = delta, displaybar=False)
    plt.title(titlestr, fontproperties = 'Times New Roman', fontsize = tsize, fontweight="bold")
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.yticks(size = ysize);
    plt.yticks(rotation=0) 
    
    ############### heat2 ############
    plt.subplot(grid[12:24,:])
    plt.subplot(3,1,2)
    h2 = topic_Change_heatmap(smt.iloc[1:,:], paraSets, delta = delta, displaybar=False)
    plt.title('')
    plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    plt.yticks(size = ysize);
    plt.yticks(rotation=0) 
    
    ############### kld trend ############
    plt.subplot(grid[24:36,:])
    h3 = plot_KLDtrend( KLDSer , paraSets, step = step)
    plt.title('')
    M = float(KLDSer.max())+0.005
    m = float(KLDSer.min())
    k = (M-m)/10
    plt.yticks(np.arange(m,M,k))

    return KLDSer , fig

In [ ]:
# Combine data
def CombineData(DocNum,splst,cplst,days):
    df = pd.DataFrame(index=days[1:])
    for sp in splst:
        for cp in cplst:
            col = 'KLD_s%d_c%d'%(sp,cp)
            df[col] = list(pd.read_csv('./Data/CBS_KDKA/LDA_KLD/Doc%d/Doc%d_smoothPeriod_%d_calPeriod_%d.csv'%(DocNum,DocNum,sp,cp), encoding = "utf_8_sig" ).iloc[:,1])
    df.to_csv('./Data/CBS_KDKA/LDA_KLD/Doc%d/Doc%d_Combined.csv'%(DocNum,DocNum), encoding = "utf_8_sig" ,  index = True)
    return df

In [ ]:
DocNum = 4
smooth_period = 14
fig = plt.figure( figsize = (12,3) ) 
smt = ForwardSmoothDist(  tfs[DocNum-1], LDAModels[DocNum-1], topicNums[DocNum-1], days, smooth_period )
topic_Change_heatmap( smt, [DocNum,smooth_period], delta = 0.1)
plt.yticks(rotation=0) 
if save_plot :
       fig.savefig(r'./Plot/CBS_KDKA/topic_trending_smooth_heatmap4.jpg', bbox_inches = 'tight')

## Calculation

In [ ]:
'''
ForwardSmoothDist(  tf, model, topicNum, days, smooth_period )
KLDSer = KLDseries( dtp, period = 7 )
topic_Change_heatmap( dtp, paraSets, delta = 0.1  )
plot_KLDtrend( KLDSer, paraSets )
KLDSer , fig = displayKLD( DocNum, smooth_period, cal_period )
'''

DocNum = 4
smooth_period = 21  # >= 0 or -1  # 14, 21  larger parameter -> flatter curve, the last two peack can be eliminated
cal_period = -1      # > 0  or -1  # 7       smaller parameter -> more peaks
paraSets = [DocNum,smooth_period,cal_period]

KLDSer , fig = displayKLD( DocNum, smooth_period, cal_period, para = [13, 18, 0.1, 5], figsize = (12,8) )

if save_plot :
    fig.savefig(r'./Data/CBS_KDKA/LDA_KLD/Doc%d/noveltyTrend%d_smoothPeriod_%d_calPeriod_%d.jpg'%(DocNum,DocNum,smooth_period,cal_period), bbox_inches = 'tight')
    KLDSer.to_csv('./Data/CBS_KDKA/LDA_KLD/Doc%d/Doc%d_smoothPeriod_%d_calPeriod_%d.csv'%(DocNum,DocNum,smooth_period,cal_period), encoding = "utf_8_sig" ,  index = True)

In [ ]:
# Combined data 4
combined4 = CombineData(DocNum=4,splst=[-1,7,14,21],cplst=[-1,7,14],days=days)

Topic1: School & University Life

Topic2: Outbreak, Close & Gov Policy

Topic3: Test count & Work

Topic4: Reopen & Safety & Restaurant

In [ ]:
DocNum = 1
smooth_period = 7  # >= 0 or -1
cal_period = 14      # > 0  or -1
paraSets = [DocNum,smooth_period,cal_period]

KLDSer , fig = displayKLD( DocNum, smooth_period, cal_period, para = [13, 18, 0.1, 5], figsize = (12,8) )
if save_plot :
       fig.savefig(r'./Plot/CBS_KDKA/noveltyTrend%d_smoothPeriod_%d_%d.jpg'%(DocNum,smooth_period,cal_period), bbox_inches = 'tight')

In [ ]:
DocNum = 3
smooth_period = -1  # >= 0 or -1
cal_period = 7      # > 0  or -1
paraSets = [DocNum,smooth_period,cal_period]

KLDSer , fig = displayKLD( DocNum, smooth_period, cal_period, para = [13, 18, 0.1, 5], figsize = (12,8) )
if save_plot :
       fig.savefig(r'./Plot/CBS_KDKA/noveltyTrend%d_smoothPeriod_%d_%d.jpg'%(DocNum,smooth_period,cal_period), bbox_inches = 'tight')

In [ ]:
DocNum = 2
smooth_period = 7  # >= 0 or -1
cal_period = 10      # > 0  or -1
paraSets = [DocNum,smooth_period,cal_period]

KLDSer , fig = displayKLD( DocNum, smooth_period, cal_period, para = [13, 18, 0.1, 5], figsize = (12,8) )
if save_plot :
       fig.savefig(r'./Plot/CBS_KDKA/noveltyTrend%d_smoothPeriod_%d_%d.jpg'%(DocNum,smooth_period,cal_period), bbox_inches = 'tight')

# Dynamic Topic Model

In [ ]:
# https://markroxor.github.io/gensim/static/notebooks/ldaseqmodel.html
import gensim
from gensim import corpora
from gensim.matutils import hellinger
from gensim.corpora import Dictionary, bleicorpus
from gensim.models import ldaseqmodel, CoherenceModel, LdaModel 

In [ ]:
# Global stopwords for DTM

stpw = ['county','pennsylvania','virginia','additional','ohio','bringing',
             'week','year','fall','countywide','area','state','report','statewide',
            'amid', 'pittsburgh','covid','district','announces','west','local','center',
            'day','phase','virus','plan','beaver','pandemic','north','washington','link',
            'july','april','announced','percent','reporting','probable','month','thing','lot',
            'hour','hill','penn','city','bull','wear','wearing','staff','including','told',
            'august','decision','group','bar','high','stand','increase','june','needed','start',
            'today','feel','pennsylvanian','team','entire']

def stpw_filter(arclst):
    global stpw
    return [ word for word in arclst if word.lower() not in stpw ] 

In [ ]:
# Prepare
# Get DTM data
cbs = pd.read_csv('./Data/CBS_KDKA/cbs_clean.csv', encoding = "utf_8_sig" )
cbs = cbs[['Date','Title','Keywords','Content']].sort_values('Date').reset_index(drop=True)

days = cbs['Date'] 
docs1 = list(cbs['Title'])
docs2 = list(cbs['Keywords'])
docs3 = list(cbs['Content'])
docs4 = [' '.join([docs1[i],docs2[i]]) for i in range(len(docs1))]

# timeslice
time_slices = []

# prepared docs
docs = [docs1, docs2, docs3, docs4] 
docs = [ [ stpw_filter(article.split(' ')) for article in doc ] for doc in docs ] 

# id2words
id2words = [ Dictionary(doc) for doc in docs ] 
for id2word in id2words:
    id2word.filter_extremes(no_below = 10 , no_above = 0.8)

# corpuses
corpuses = [ [id2words[i].doc2bow(doc) for doc in docs[i]] for i in range(4)]

# docs[i] <--> id2words[i] <--> corpuses[i]

In [ ]:
# Check for empty docs
# https://stackoverflow.com/questions/63100943/ldaseqmodel-runtimewarning-invalid-value-in-double-scalars

empty_doc_index_lsts = []
for i in range(4):
    print('Start check docs: ',(i+1))
    empty_doc_index = []
    for j in range(len(corpuses[i])): 
        if len(corpuses[i][j])==0:    # check for empty document
            empty_doc_index.append(j) 
    empty_doc_index_lsts.append(empty_doc_index)
    print(empty_doc_index)
    print('Total %d docs out of 2066'%len(empty_doc_index))
    print('End check docs: ',(i+1))
    print('')

In [ ]:
from analysis_utils import prepare_dtm
docs, corpuses, time_slices, day_lsts = prepare_dtm(docs, corpuses, cbs["Date"])


In [ ]:
# Double Check
empty_doc_index_lsts = []
for i in range(4):
    print('Start check docs: ',(i+1))
    empty_doc_index = []
    for j in range(len(corpuses[i])): 
        if len(corpuses[i][j])==0:    # check for empty document
            empty_doc_index.append(j) 
    empty_doc_index_lsts.append(empty_doc_index)
    print(empty_doc_index)
    print('Total %d docs out of 2066'%len(empty_doc_index))
    print('End check docs: ',(i+1))
    print('')
    
# Check for time_slices: sum(time_slices[i]) == len(docs[i])
for i in range(4):
    print(sum(time_slices[i])==len(docs[i]))

In [ ]:
%%time
# Topic Nums
topicNums = [5,5,5,5]
dtms = []

for i in tqdm(range(4)):
    dtm = ldaseqmodel.LdaSeqModel(
        corpus = corpuses[i], 
        id2word = id2words[i], 
        time_slice = time_slices[i], 
        num_topics = topicNums[i])
    dtms.append(dtm)

In [ ]:
%%time
# Topic Nums

i = 0
k = 100
testtime_slice = time_slices[i][:k]
j = sum(testtime_slice)
testdoc = docs[i][:j]
testcorpus = corpuses[i][:j]


dtm = ldaseqmodel.LdaSeqModel(
        corpus = testcorpus, 
        id2word = id2words[i], 
        time_slice = testtime_slice, 
        num_topics = 5)


In [ ]:
sum(time_slices[i][:20])
# 256 : 42s

In [ ]:
len(time_slices[i])

In [ ]:
# Double Check
print('Start check docs')
empty_doc_index = []
for j in range(len(testcorpus)): 
    if len(testcorpus[j])==0:    # check for empty document
        empty_doc_index.append(j) 
empty_doc_index_lsts.append(empty_doc_index)
print(empty_doc_index)
print('Total %d docs out of 2066'%len(empty_doc_index))
print('End check docs: ',(i+1))
print('')